# Limpieza EC CHM 2026

Limpieza del export de Qualtrics (*Copia de Instrumento EC CHM 2026*). Produce cuatro archivos:

- **Docentes únicos**: un registro por cédula (CHM y DA), con cédula y nombre separados y normalizados.
- **Momentos**: una fila por docente y por afirmación, con la respuesta y el comentario de cada momento.
- **Cierre de momentos**: las 5 preguntas de cierre de cada momento.
- **Instantáneas**: una fila por instantánea con contenido (hasta 18 por respuesta).

`fila_origen` es la fila de la hoja original y `columnas_origen` las letras de columna, para comprobar cada dato en Google Sheets.

## Lectura y utilidades

In [1]:
import re
import pandas as pd
from openpyxl.utils import column_index_from_string as ci, get_column_letter as letra

SHEET_ID = "1uKoYzNM0I-XvD3N8Q8u6aOngyozCfkj5s3rHTY8_XZ8"
GID = "437169291"
URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"

df = pd.read_csv(URL, dtype=str)
COL_ID = "ID de respuesta"
assert df.columns[ci("I") - 1] == COL_ID

In [2]:
N = 408                                                       # columnas por momento
INICIO = {1: "BE", 2: "AJO", 3: "AZG", 4: "BOY", 5: "CEQ"}   # primera columna de cada momento
PARTICULAS = {"de", "del", "la", "las", "los", "y", "e"}

def rango(a, b):
    return list(range(ci(a) - 1, ci(b)))

def limpio_texto(s):
    if pd.isna(s):
        return None
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s or None

def limpio_encabezado(s):
    return limpio_texto(str(s).replace("💡", "").replace("\ufe0f", ""))

def normaliza_nombre(s):
    s = limpio_texto(s)
    if s is None:
        return None
    if s.casefold() == "no aplica":
        return "No aplica"
    palabras = s.casefold().split(" ")
    return " ".join(p if (i > 0 and p in PARTICULAS) else p[:1].upper() + p[1:]
                    for i, p in enumerate(palabras))

## Docentes

In [3]:
# Docentes por respuesta: CHM1-5 (V:Z) y DA1-10 (AU:BD), cédula y nombre separados y normalizados
CODIGOS = [f"CHM{i}" for i in range(1, 6)] + [f"DA{i}" for i in range(1, 11)]

roster = df.iloc[:, rango("V", "Z") + rango("AU", "BD")].copy()
roster.columns = CODIGOS
roster.insert(0, "id_respuesta", df[COL_ID].values)
roster.insert(0, "fila_origen", df.index + 2)
roster = roster.melt(id_vars=["fila_origen", "id_respuesta"], var_name="codigo", value_name="raw")
roster["raw"] = roster["raw"].astype("string").str.strip()
roster = roster[roster["raw"].notna() & (roster["raw"] != "")]

ext = roster["raw"].str.extract(r"^(\d[\d\.]*)\s*[-–:]?\s*(.+)$")
roster["cedula"] = ext[0].str.replace(".", "", regex=False)
roster["nombre"] = ext[1].fillna(roster["raw"]).map(normaliza_nombre)   # sin cédula: queda el texto ("No aplica")
roster["rol"] = roster["codigo"].str.extract(r"^([A-Z]+)")[0]
roster["_o"] = roster["codigo"].map({c: i for i, c in enumerate(CODIGOS)})
roster = roster.sort_values(["fila_origen", "_o"]).reset_index(drop=True)
print(roster.shape)

(85, 8)


In [4]:
# Una vez cada cédula; las filas sin cédula ("No aplica", etc.) se conservan todas
tiene_cedula = roster["cedula"].notna()
df_docentes_unicos = roster[~tiene_cedula | ~roster["cedula"].duplicated()].copy()
df_docentes_unicos["sexo"] = None                  # se completa a mano en otro documento
df_docentes_unicos = df_docentes_unicos[
    ["id_respuesta", "codigo", "rol", "cedula", "nombre", "sexo"]].reset_index(drop=True)

print(len(roster), "->", len(df_docentes_unicos), "| sin cédula:", (~tiene_cedula).sum())

85 -> 85 | sin cédula: 22


## Momentos

La plantilla se lee del Momento 1 y se aplica por posición a los 5 momentos.

In [5]:
# La plantilla se lee de los encabezados del Momento 1 y se aplica por posición a los 5 momentos
BLOQUES = {
    "Respuestas": "Respuesta", "Respuesta": "Respuesta",
    "Tipo de docente": "Tipo", "Tipo de docente implicado": "Tipo",
    "ID del/los docentes": "ID", "Comentarios adicionales": "Comentario",
}
GRUPOS, CIERRE = {}, []
ini = ci(INICIO[1]) - 1
for pos, h in enumerate(df.columns[ini : ini + N]):
    p = [limpio_encabezado(x) for x in re.sub(r"\.\d+$", "", limpio_encabezado(h)).split(" - ")]
    if p[0].startswith("Timing"):
        continue                                   # los 4 Timing se descartan
    if len(p) >= 3 and p[1] in BLOQUES:
        campo = BLOQUES[p[1]]
        g = GRUPOS.setdefault((p[0], p[2]), {"Respuesta": None, "Comentario": None, "ID": {}})
        if campo == "ID":
            g["ID"][p[3]] = pos                    # p[3] es el código: CHM1 ... DA10
        elif campo != "Tipo":                      # el rol sale del código, no de "Tipo de docente"
            g[campo] = pos
    else:
        CIERRE.append((pos, p[0]))                 # preguntas de cierre

assert len(GRUPOS) == 21 and len(CIERRE) == 5, (len(GRUPOS), len(CIERRE))
print(len(GRUPOS), "afirmaciones;", len(CIERRE), "preguntas de cierre")

21 afirmaciones; 5 preguntas de cierre


In [6]:
# Datos de la respuesta: observador primero, luego los docentes
meta = pd.DataFrame({
    "fila_origen": df.index + 2,                   # fila de la hoja original (fila 1 = encabezados)
    "id_respuesta": df[COL_ID].values,
    "observador": df.iloc[:, ci("R") - 1].map(normaliza_nombre).values,
    "coding_hub": df.iloc[:, ci("U") - 1].map(limpio_texto).values,
    "instituciones": df.iloc[:, rango("AA", "AT")].apply(
        lambda f: " | ".join(dict.fromkeys(v for v in map(limpio_texto, f) if v)) or None,
        axis=1).values,
})

partes = []
for m, inicio in INICIO.items():
    c0 = ci(inicio)
    x = df.iloc[:, c0 - 1 : c0 - 1 + N]
    for orden_af, ((cat, af), g) in enumerate(GRUPOS.items()):
        base = meta.copy()
        base["momento"] = f"Momento {m}"
        base["categoria"] = cat
        base["afirmacion"] = af
        base["respuesta"] = x.iloc[:, g["Respuesta"]].str.strip().values
        base["comentario"] = x.iloc[:, g["Comentario"]].values
        col_resp, col_com = letra(c0 + g["Respuesta"]), letra(c0 + g["Comentario"])

        ids = x.iloc[:, list(g["ID"].values())].copy()
        ids.columns = list(g["ID"].keys())
        marcado = ids.notna() & (ids.astype("string").apply(lambda s: s.str.strip()) != "")

        for orden_cod, (cod, pos) in enumerate(g["ID"].items()):
            p = base[marcado[cod].values].copy()
            p["codigo"] = cod
            p["columnas_origen"] = f"Respuesta {col_resp} | ID {letra(c0 + pos)} | Comentario {col_com}"
            p["_o"] = orden_af * 100 + orden_cod
            partes.append(p)

        sin = base[~marcado.any(axis=1).values].copy()   # afirmación sin docente marcado: se conserva
        sin["columnas_origen"] = f"Respuesta {col_resp} | Comentario {col_com}"
        sin["_o"] = orden_af * 100 + 99
        partes.append(sin)

COLS_MOMENTOS = ["fila_origen", "id_respuesta", "observador", "coding_hub", "instituciones",
                 "momento", "categoria", "afirmacion", "respuesta", "comentario",
                 "rol", "codigo", "cedula", "nombre", "columnas_origen"]

df_momentos = pd.concat(partes, ignore_index=True)
df_momentos["rol"] = df_momentos["codigo"].str.extract(r"^([A-Z]+)")[0]
df_momentos = df_momentos.merge(roster[["id_respuesta", "codigo", "cedula", "nombre"]],
                                on=["id_respuesta", "codigo"], how="left")
df_momentos = (df_momentos.sort_values(["fila_origen", "momento", "_o"], kind="stable")
               [COLS_MOMENTOS].reset_index(drop=True))
print(df_momentos.shape)

(1853, 15)


## Cierre de momentos

In [7]:
cierre = []
for m, inicio in INICIO.items():
    c0 = ci(inicio)
    x = df.iloc[:, c0 - 1 : c0 - 1 + N]
    t = meta[["fila_origen", "id_respuesta"]].copy()
    t["momento"] = f"Momento {m}"
    for pos, nombre in CIERRE:
        t[nombre] = x.iloc[:, pos].values
    cierre.append(t)

df_cierre = (pd.concat(cierre, ignore_index=True)
             .sort_values(["fila_origen", "momento"], kind="stable")
             .reset_index(drop=True))
df_cierre = df_cierre[["fila_origen", "id_respuesta", "momento"] + [n for _, n in CIERRE]]
print(df_cierre.shape)

(40, 8)


## Instantáneas

In [8]:
# 18 instantáneas de 27 columnas (QW:AJN)
COLS_INST = [
    "fila_origen", "id_respuesta", "observador", "coding_hub", "numero_instantanea",
    "accion_momento", "accion_momento_otra",
    "participa_coding_hub_masters", "participa_docentes_acompanados", "participa_mentores",
    "participa_no_aplica", "participa_hombres", "participa_mujeres", "participa_genero_no_aplica",
    "quien_dirige", "en_que_momento",
]

inst = df.iloc[:, rango("QW", "AJN")]

def limpia(s):
    return s.map(limpio_texto)

def unir(fila):
    return "; ".join(dict.fromkeys(v for v in map(limpio_texto, fila) if v))

partes = []
for k in range(18):
    b = inst.iloc[:, 27 * k : 27 * (k + 1)]
    out = meta[["fila_origen", "id_respuesta", "observador", "coding_hub"]].copy()
    out["numero_instantanea"]             = k + 1
    out["accion_momento"]                 = b.iloc[:, 0:17].apply(unir, axis=1).values
    out["accion_momento_otra"]            = limpia(b.iloc[:, 17]).values
    out["participa_coding_hub_masters"]   = limpia(b.iloc[:, 18]).values
    out["participa_docentes_acompanados"] = limpia(b.iloc[:, 19]).values
    out["participa_mentores"]             = limpia(b.iloc[:, 20]).values
    out["participa_no_aplica"]            = limpia(b.iloc[:, 21]).values
    out["participa_hombres"]              = limpia(b.iloc[:, 22]).values
    out["participa_mujeres"]              = limpia(b.iloc[:, 23]).values
    out["participa_genero_no_aplica"]     = limpia(b.iloc[:, 24]).values
    out["quien_dirige"]                   = limpia(b.iloc[:, 25]).values
    out["en_que_momento"]                 = limpia(b.iloc[:, 26]).values
    partes.append(out)

df_instantaneas = pd.concat(partes, ignore_index=True)

# se conserva la instantánea solo si tiene contenido
tiene = ((df_instantaneas["accion_momento"] != "")
         | df_instantaneas["en_que_momento"].notna()
         | df_instantaneas["quien_dirige"].notna())
df_instantaneas = (df_instantaneas[tiene][COLS_INST]
                   .sort_values(["fila_origen", "numero_instantanea"])
                   .reset_index(drop=True))
print(df_instantaneas.shape)

(114, 16)


## Exportación

In [9]:
SALIDAS = [
    (df_docentes_unicos, "Docentes_unicos_EC_CHM_2026_0825.csv"),
    (df_momentos,        "Momentos_EC_CHM_2026_0825.csv"),
    (df_cierre,          "Cierre_Momentos_EC_CHM_2026_0825.csv"),
    (df_instantaneas,    "Instantaneas_EC_CHM_2026_0825.csv"),
]
for d, nombre in SALIDAS:
    d.to_csv(nombre, index=False, encoding="utf-8-sig")

try:
    from google.colab import files
    for _, nombre in SALIDAS:
        files.download(nombre)
except ImportError:
    print("Guardados en la carpeta actual:", ", ".join(n for _, n in SALIDAS))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>